In [3]:
# 🎬 Movie Recommendation System (Content-Based)

# ------------------ IMPORTS ------------------
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import importlib
import recommender
importlib.reload(recommender)

from recommender import recommend

# ------------------ LOAD DATA ------------------

# If using CSV file
movies = pd.read_csv("movies.csv")

# Fill missing values
movies["genre"] = movies["genre"].fillna("")
movies["overview"] = movies["overview"].fillna("")

# Combine features
movies["combined_features"] = movies["genre"] + " " + movies["overview"]


# ------------------ FEATURE ENGINEERING ------------------

# Convert text to numerical vectors using TF-IDF
tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(movies["combined_features"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)


# ------------------ SIMILARITY ------------------

# Compute cosine similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)


# ------------------ HELPER ------------------

# Create title → index mapping
movies["title_lower"] = movies["title"].str.lower()
title_to_index = pd.Series(movies.index, index=movies["title_lower"])


# ------------------ RECOMMEND FUNCTION ------------------
def recommend(movie_name, n_recommendations=5):
    movie_name = movie_name.lower()

    if movie_name not in title_to_index:
        corrected = get_closest_title(movie_name)
        print(f"Did you mean: {corrected}?")
        movie_name = corrected.lower()

    idx = title_to_index[movie_name]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:n_recommendations + 1]

    movie_indices = [i[0] for i in sim_scores]

    return movies["title"].iloc[movie_indices].tolist()


# ------------------ TEST ------------------

print("\nAvailable Movies:")
print(movies["title"].tolist())

print("\nRecommendations for 'Inception':")
print(recommend("Inception"))

print("\nRecommendations for 'The Matrix':")
print(recommend("The Matrix"))




FileNotFoundError: [Errno 2] No such file or directory: 'movies.csv'

In [ ]:
recommend("Inception")




['The Matrix', 'The Prestige', 'Interstellar', 'Memento', 'Avatar']

In [ ]:
recommend("incption")

Movie 'incption' not found.
Available movies:
['The Dark Knight', 'Inception', 'Interstellar', 'The Matrix', 'The Prestige', 'Memento', 'Avatar']


[]

In [ ]:
from rapidfuzz import process

def get_closest_title(name):
    match = process.extractOne(name, movies["title"])
    return match[0]


In [4]:
recommend("incption")


['The Matrix (score: 0.19)',
 'The Prestige (score: 0.13)',
 'Interstellar (score: 0.11)',
 'Memento (score: 0.1)',
 'Avatar (score: 0.05)']

In [5]:
from recommender import recommend

recommend("incption")

['The Matrix (score: 0.19)',
 'The Prestige (score: 0.13)',
 'Interstellar (score: 0.11)',
 'Memento (score: 0.1)',
 'Avatar (score: 0.05)']